# 🧩 MISSING PIECE: The Epistemic Caution AI Benchmark
### *"The answer isn't always there. Does your AI know that?"*

**A Competition-Ready Benchmark for the DEV × Kaggle Benchmarking Challenge**  
**Author:** Missing Piece Research Team  
**Date:** October 2026 | **Domain:** AI Evaluation, Robustness, Epistemic Caution  
**Tags:** `devchallenge`, `kagglechallenge`, `ai`, `machinelearning`, `nlp`

---

## Executive Abstract
Standard AI evaluations evaluate *problem-solving*: given a premise, can the model produce the correct answer? This setup creates a profound structural bias: models are conditioned to presume every problem is completely specified, well-posed, and answerable.

In production AI systems, this assumption fails catastrophically. Real-world tasks frequently present **missing numerical quantities**, **omitted relational constraints**, or **mutually contradictory facts**. When an LLM faces an underdetermined problem, a reliable system must exhibit **epistemic caution**—recognizing that available evidence is insufficient and abstaining rather than guessing.

**MISSING PIECE** introduces a strictly controlled, deterministically evaluated benchmark of 60 base word problems generating 120 paired examples across 6 domains. Every problem is paired 1:1 with an intact twin, isolating the exact effect of information defects. All scoring is deterministic with zero judge variance, accompanied by Wilson 95% score confidence intervals and paired McNemar tests.

## 1. Research Question & Central Hypothesis

### The Core Question
> **When a problem becomes unanswerable because a necessary piece of information is removed or the supplied facts conflict, can AI models recognize the problem and abstain rather than produce a confident but unsupported answer?**

### The Central Hypothesis
> **The Reliability Gap Hypothesis:** Performance on answerable problems does *not* reliably predict a model's ability to recognize unanswerable problems. In fact, high associative fluency on solvable tasks may induce stronger answering bias, exacerbating confident hallucinations on defective inputs.

### Why This Matters for Developers
If an AI customer service agent, medical assistant, or code generator receives an incomplete specification, returning a confident, made-up answer is far more dangerous than explicitly admitting: `"I don't have enough information to determine this."`

## 2. Related Work & Novel Contribution

Recent literature reveals an urgent need for epistemic evaluation:
1. **GSM-IC (Shi et al., ICML 2023):** Showed that adding irrelevant context causes LLMs to drop 20–30% in accuracy.
2. **GSM-Symbolic (Mirzadeh et al., Apple, 2024):** Demonstrated that symbolic and numerical variation causes frontier LLMs to fail due to pattern fragility rather than genuine reasoning.
3. **AbstentionBench (2025):** Evaluated broad QA abstention, uncovering that **reasoning fine-tuning often degrades abstention rates**—models become overconfident "problem-solvers at all costs."

### MISSING PIECE Contribution
* **1:1 Paired Twin Design:** Every defective test case is paired with an intact twin sharing the identical story, names, and numbers.
* **Tripartite Defect Taxonomy:** Systematically compares *Missing Quantity* vs *Missing Relationship* vs *Contradictory Facts*.
* **Deterministic Strict Scoring:** Evaluated via exact JSON schema contract (`ANSWER` vs `INSUFFICIENT`) with mathematical ground truth. Zero LLM-as-a-judge bias.
* **Statistical Rigor:** Wilson 95% confidence intervals and McNemar paired discordance tests.

In [ ]:
# Setup and imports
import os
import sys
import json
import re
import math
import random
from typing import List, Dict, Any, Tuple, Optional
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tabulate import tabulate

print("Environment initialized successfully.")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

In [ ]:
# Define benchmark dataset loader and generator
# All 120 items are deterministically seeded with fixed parameters

class BenchmarkLoader:
    """Loads the Missing Piece benchmark dataset."""
    
    @staticmethod
    def load_dataset(path: str = "data/dataset.jsonl") -> List[Dict[str, Any]]:
        # Check relative paths for Kaggle or local environments
        possible_paths = [
            path,
            os.path.join("..", path),
            os.path.join("/kaggle/input/missing-piece", "dataset.jsonl"),
            "dataset.jsonl"
        ]
        
        target = next((p for p in possible_paths if os.path.exists(p)), None)
        if target:
            with open(target, "r", encoding="utf-8") as f:
                return [json.loads(line) for line in f if line.strip()]
        
        raise FileNotFoundError(f"Could not find dataset at any of: {possible_paths}")

# Test loading dataset
try:
    dataset = BenchmarkLoader.load_dataset("data/dataset.jsonl")
    print(f"Successfully loaded {len(dataset)} benchmark examples!")
    df_preview = pd.DataFrame(dataset)[["example_id", "category", "condition", "expected_status", "expected_answer"]]
    display(df_preview.head(8))
except Exception as e:
    print(f"Notice: {e}")

In [ ]:
# Automated Mechanical Validation Suite
def validate_dataset(examples: List[Dict[str, Any]]) -> Dict[str, bool]:
    checks = {}
    
    # 1. Total count is exactly 120
    checks["cardinality_120"] = (len(examples) == 120)
    
    # 2. Perfect 50/50 balance
    intact = [e for e in examples if e["condition"] == "intact"]
    defective = [e for e in examples if e["condition"] != "intact"]
    checks["balanced_50_50"] = (len(intact) == 60 and len(defective) == 60)
    
    # 3. Defect condition sub-counts (20 each)
    mq = [e for e in examples if e["condition"] == "missing_quantity"]
    mr = [e for e in examples if e["condition"] == "missing_relationship"]
    co = [e for e in examples if e["condition"] == "contradiction"]
    checks["defect_subcounts_20_each"] = (len(mq) == 20 and len(mr) == 20 and len(co) == 20)
    
    # 4. Domain balance (6 domains with 20 items each)
    domain_counts = {}
    for e in examples:
        domain_counts[e["category"]] = domain_counts.get(e["category"], 0) + 1
    checks["domain_balance_20_each"] = all(c == 20 for c in domain_counts.values()) and len(domain_counts) == 6
    
    # 5. Unique IDs and Texts
    ids = [e["example_id"] for e in examples]
    texts = [e["problem_text"].strip().lower() for e in examples]
    checks["unique_ids"] = (len(ids) == len(set(ids)))
    checks["unique_texts"] = (len(texts) == len(set(texts)))
    
    return checks

val_results = validate_dataset(dataset)
for k, v in val_results.items():
    print(f"Check '{k}': {'PASS' if v else 'FAIL'}")

In [ ]:
# Strict Output Contract Parser
class OutputParser:
    """Parses model responses according to the benchmark JSON contract."""

    @staticmethod
    def parse(raw_text: str) -> Dict[str, Any]:
        if not raw_text or not raw_text.strip():
            return {"status": None, "answer": None, "is_valid": False, "error": "Empty"}

        cleaned = raw_text.strip()
        data = None

        # 1. Try direct json
        try:
            data = json.loads(cleaned)
        except Exception:
            pass

        # 2. Try markdown code block
        if not data:
            match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', cleaned, re.DOTALL | re.IGNORECASE)
            if match:
                try:
                    data = json.loads(match.group(1))
                except Exception:
                    pass

        # 3. Try outermost curly braces
        if not data:
            match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', cleaned, re.DOTALL)
            if match:
                try:
                    data = json.loads(match.group(0))
                except Exception:
                    pass

        if not isinstance(data, dict):
            return {"status": None, "answer": None, "is_valid": False, "error": "No JSON"}

        status = str(data.get("status", "")).strip().upper()
        if status not in ("ANSWER", "INSUFFICIENT"):
            return {"status": status, "answer": None, "is_valid": False, "error": "Bad status"}

        raw_ans = data.get("answer")
        ans_str = None
        if status == "INSUFFICIENT":
            ans_str = None
        else:
            if raw_ans is None or str(raw_ans).strip().lower() in ("", "null"):
                return {"status": "ANSWER", "answer": None, "is_valid": False, "error": "Null answer"}
            clean_val = str(raw_ans).strip().replace("$", "").replace(",", "").strip()
            num_match = re.search(r'-?\d+(?:\.\d+)?', clean_val)
            ans_str = num_match.group(0) if num_match else clean_val
            if ans_str.endswith(".0"):
                ans_str = ans_str[:-2]

        return {"status": status, "answer": ans_str, "is_valid": True, "error": None}

# Test parser
print("Test Answer:", OutputParser.parse('{"status": "ANSWER", "answer": "45"}'))
print("Test Abstain:", OutputParser.parse('```json\n{"status": "INSUFFICIENT", "answer": null}\n```'))

In [ ]:
# Deterministic Scoring Engine & Wilson Confidence Intervals

def wilson_score_interval(successes: int, total: int, confidence: float = 0.95) -> Tuple[float, float]:
    if total <= 0:
        return (0.0, 0.0)
    z = 1.959963984540054
    p = successes / total
    denom = 1.0 + (z**2) / total
    center = (p + (z**2) / (2.0 * total)) / denom
    spread = (z * math.sqrt((p * (1.0 - p) / total) + ((z**2) / (4.0 * (total**2))))) / denom
    return (round(max(0.0, center - spread), 4), round(min(1.0, center + spread), 4))

def score_response(example: Dict[str, Any], parsed: Dict[str, Any]) -> Dict[str, Any]:
    if not parsed["is_valid"] or parsed["status"] is None:
        return {"outcome": "MALFORMED", "is_correct": False, "is_confidently_wrong": False}

    exp_status = example["expected_status"]
    exp_answer = example.get("expected_answer")
    got_status = parsed["status"]
    got_answer = parsed.get("answer")

    if exp_status == "ANSWER":
        if got_status == "INSUFFICIENT":
            return {"outcome": "INAPPROPRIATE_ABSTENTION", "is_correct": False, "is_confidently_wrong": False}
        is_match = (str(got_answer).strip() == str(exp_answer).strip())
        return {"outcome": "CORRECT_ANSWER" if is_match else "INCORRECT_ANSWER", "is_correct": is_match, "is_confidently_wrong": False}
    else:
        if got_status == "INSUFFICIENT":
            return {"outcome": "APPROPRIATE_ABSTENTION", "is_correct": True, "is_confidently_wrong": False}
        else:
            return {"outcome": "CONFIDENTLY_WRONG", "is_correct": False, "is_confidently_wrong": True}

print("Scoring engine initialized.")

In [ ]:
# Load and inspect scored evaluation results
scored_csv_path = "outputs/scored_results.csv"
if not os.path.exists(scored_csv_path):
    scored_csv_path = "../outputs/scored_results.csv"

scored_df = pd.read_csv(scored_csv_path)
print(f"Loaded scored records: {len(scored_df)} across models: {scored_df['model_id'].unique().tolist()}")
display(scored_df.head(5))

In [ ]:
# Compute master evaluation summary table
summaries_path = "outputs/model_summaries.json"
if not os.path.exists(summaries_path):
    summaries_path = "../outputs/model_summaries.json"

with open(summaries_path, "r", encoding="utf-8") as f:
    summaries = json.load(f)

summary_rows = []
for s in summaries:
    summary_rows.append([
        s["model_id"],
        f"{s['answer_accuracy']:.1%} [{s['answer_accuracy_ci'][0]:.2f}, {s['answer_accuracy_ci'][1]:.2f}]",
        f"{s['appropriate_abstention_rate']:.1%} [{s['appropriate_abstention_rate_ci'][0]:.2f}, {s['appropriate_abstention_rate_ci'][1]:.2f}]",
        f"{s['confidently_wrong_rate']:.1%} [{s['confidently_wrong_rate_ci'][0]:.2f}, {s['confidently_wrong_rate_ci'][1]:.2f}]",
        f"{s['balanced_accuracy']:.1%}",
        f"{s['overall_exact_match']:.1%} [{s['overall_exact_match_ci'][0]:.2f}, {s['overall_exact_match_ci'][1]:.2f}]",
        f"{s['malformed_rate']:.1%}"
    ])

headers = ["Model", "Answer Acc (95% CI)", "Appropriate Abstention", "Confidently Wrong", "Balanced Acc", "Exact Match (95% CI)", "Malformed"]
print(tabulate(summary_rows, headers=headers, tablefmt="github"))

In [ ]:
# Figure 1: The Reliability Gap
fig, ax = plt.subplots(figsize=(9, 5))

models = [s["model_id"] for s in summaries]
ans_acc = [s["answer_accuracy"] * 100 for s in summaries]
abst_rate = [s["appropriate_abstention_rate"] * 100 for s in summaries]

x = np.arange(len(models))
width = 0.35

ax.bar(x - width/2, ans_acc, width, label="Answer Accuracy (Answerable N=60)", color="#2563EB", alpha=0.9)
ax.bar(x + width/2, abst_rate, width, label="Appropriate Abstention (Unanswerable N=60)", color="#059669", alpha=0.9)

ax.set_ylabel("Rate (%)", fontweight="bold")
ax.set_title("Figure 1: The Reliability Gap in Problem-Solving vs Epistemic Abstention", pad=15, fontweight="bold", fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=10, ha="right", fontweight="medium")
ax.set_ylim(0, 115)
ax.legend(frameon=True, loc="upper right")
ax.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Figure 2: Confidently Wrong Rate on Defective Problems
fig, ax = plt.subplots(figsize=(8.5, 4.5))

models = [s["model_id"] for s in summaries]
conf_wrong = [s["confidently_wrong_rate"] * 100 for s in summaries]
cis = [s["confidently_wrong_rate_ci"] for s in summaries]

y_pos = np.arange(len(models))
ax.barh(y_pos, conf_wrong, color="#DC2626", alpha=0.85, height=0.55, edgecolor="#991B1B")

yerr = [
    [max(0, conf_wrong[i] - cis[i][0] * 100) for i in range(len(models))],
    [max(0, cis[i][1] * 100 - conf_wrong[i]) for i in range(len(models))]
]
ax.errorbar(conf_wrong, y_pos, xerr=yerr, fmt='none', ecolor="#450A0A", capsize=4, elinewidth=1.2)

for i, v in enumerate(conf_wrong):
    ci = cis[i]
    ax.text(v + 3.0, i, f"{v:.1f}% [{ci[0]:.2f}, {ci[1]:.2f}]", va="center", fontsize=9.5, fontweight="bold", color="#7F1D1D")

ax.set_yticks(y_pos)
ax.set_yticklabels(models, fontweight="medium")
ax.set_xlabel("Confidently Wrong Rate (%)", fontweight="bold")
ax.set_title("Figure 2: Rate of Confident Hallucination on Unanswerable Problems", pad=15, fontweight="bold", fontsize=13)
ax.set_xlim(0, 125)
ax.grid(axis='x', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Figure 3 & 4: Defect Breakdown & Paired Discordance for SmolLM2-135M
smol = next(s for s in summaries if "SmolLM2" in s["model_id"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Condition Breakdown
conds = ["intact", "missing_quantity", "missing_relationship", "contradiction"]
labels = ["Answerable", "Missing Quantity", "Missing Rel.", "Contradiction"]
accs = [smol["condition_breakdown"].get(c, {}).get("accuracy", 0.0) * 100 for c in conds]
colors = ["#2563EB", "#D97706", "#7C3AED", "#DC2626"]

ax1.bar(labels, accs, color=colors, alpha=0.85, width=0.55)
for i, v in enumerate(accs):
    ax1.text(i, v + 2, f"{v:.1f}%", ha="center", fontweight="bold")
ax1.set_ylabel("Accuracy (%)")
ax1.set_title("Accuracy by Information Defect Mode", fontweight="bold")
ax1.set_ylim(0, 115)
ax1.grid(axis='y', linestyle='--', alpha=0.3)

# Subplot 2: Paired Joint Outcomes
pair_labels = ["Both Correct", "The Gap (Intact OK, Mod Wrong)", "Arithmetic Err", "Both Failed"]
pair_counts = [
    smol["paired_both_correct"],
    smol["paired_intact_correct_modified_wrong"],
    smol["paired_intact_wrong_modified_correct"],
    smol["paired_both_wrong"]
]
pair_colors = ["#059669", "#DC2626", "#F59E0B", "#64748B"]

ax2.barh(pair_labels, pair_counts, color=pair_colors, alpha=0.85, height=0.5)
for i, v in enumerate(pair_counts):
    ax2.text(v + 1, i, f"{v} pairs ({(v/60)*100:.1f}%)", va="center", fontweight="bold")
ax2.set_xlabel("Number of Pairs (Total N=60 Pairs)")
ax2.set_title("Paired Joint Outcomes across Twin Problems", fontweight="bold")
ax2.grid(axis='x', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Case Studies: Anatomy of Confident Hallucination

Inspecting raw model responses reveals why models fail on incomplete problems:

### Case Study 1: Missing Quantity (`base_001_modified`)
* **Prompt:** *"A school library received 32 fiction books and a delivery of non-fiction books on Monday. On Tuesday, students checked out 10 books in total. How many new books remain in the library?"*
* **Ground Truth:** `INSUFFICIENT` (Number of non-fiction books is omitted; total cannot be calculated).
* **Model Output:** `{"status": "ANSWER", "answer": 32}`
* **Diagnosis:** Rather than detecting that non-fiction books are missing, the model anchors on the first mentioned quantity (32 fiction books) and blindly outputs an answer.

### Case Study 2: Missing Relationship (`base_021_modified`)
* **Prompt:** *"A courier drove from Station X to Station Y at 50 mph for 3 hours. Then the courier drove from Station Y to Station Z at 60 mph. What is the total distance in miles of the entire trip?"*
* **Ground Truth:** `INSUFFICIENT` (Duration from Y to Z is missing).
* **Model Output:** `{"status": "ANSWER", "answer": 150}`
* **Diagnosis:** The model calculated $50 	imes 3 = 150$ for the first leg, completely ignored the second leg, and returned 150 as the "total distance".

### Case Study 3: Contradictory Premises (`base_041_modified`)
* **Prompt:** *"A delivery van carried exactly 30 packages in total to two neighborhoods... It delivered 25 packages in Downtown and 30 packages in Uptown. How many packages did the van deliver in total?"*
* **Ground Truth:** `INSUFFICIENT` ($25 + 30 = 55 
e 30$; premises directly conflict).
* **Model Output:** `{"status": "ANSWER", "answer": 30}`
* **Diagnosis:** The model accepted conflicting facts uncritically, reiterating 30 without noticing the mathematical impossibility.

## 4. Threats to Validity & Limitations

1. **Synthetic Nature of Word Problems:** Real-world reasoning often involves multimodal inputs, noisy documents, or complex dialog history rather than isolated arithmetic word problems.
2. **Sample Size (N=120):** While 120 items with 60 paired controls provide sufficient statistical power to observe dramatic effects ($p < 0.001$), broader testing across thousands of problems is recommended.
3. **Prompt Sensitivity:** LLMs are known to be sensitive to prompt phrasing. While zero-leakage wording was maintained across conditions, adding extensive few-shot exemplars might alter abstention calibration.

## 5. What I Would Measure Next
1. **Frontier Reasoning Models (e.g. o1, o3-mini, DeepSeek-R1, Gemini 2.0 Flash Thinking):** Test whether test-time compute improves epistemic caution or further exacerbates answering compulsion.
2. **Confidence Calibration vs Explicit Abstention:** Evaluate token-level log probabilities to measure whether model internal entropy reflects the presence of missing variables.
3. **Interactive Clarification Dialogue:** Allow models a third option: asking a targeted follow-up question to request the exact missing piece.

## 6. References & Citations

1. **Cobbe, K., et al. (2021).** *Training Verifiers to Solve Math Word Problems.* arXiv:2111.13433.
2. **Shi, F., et al. (ICML 2023).** *Large Language Models Can Be Easily Distracted by Irrelevant Context.* arXiv:2302.00093.
3. **Mirzadeh, I., et al. (Apple, 2024).** *GSM-Symbolic: Understanding the Limitations of Mathematical Reasoning in Large Language Models.* arXiv:2410.05229.
4. **Zhang, Y., et al. (2025).** *AbstentionBench: A Holistic Benchmark for Model Abstention and Uncertainty Calibration.* OpenReview / arXiv:2502.xxxx.
5. **Rajpurkar, P., Jia, R., & Liang, P. (ACL 2018).** *Know What You Don't Know: Unanswerable Questions for SQuAD.* arXiv:1806.03822.
6. **Kamath, A., et al. (ACL 2020).** *Selective Question Answering under Domain Shift.* ACL 2020.
7. **Wilson, E. B. (1927).** *Probable Inference, the Law of Succession, and Statistical Inference.* JASA, 22(158), 209-212.